# 02 - Preprocessing Validation

This notebook validates the Step 3.5 preprocessing pipeline:
- Raw data loading and shape
- Duplicate removal
- Hourly aggregation
- Coverage analysis
- Quality checks

In [ ]:
import os
import sys
import numpy as np
import pandas as pd

# Ensure project root is on path
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

from src.data.load_data import load_raw_data, parse_and_sort_timestamps
from src.data.clean_data import remove_exact_duplicate_rows, inspect_duplicate_timestamps
from src.data.aggregate import (
    add_wind_circular_features, aggregate_to_hourly,
    build_complete_hourly_index, calculate_hourly_coverage
)
print('Imports successful.')

## 1. Load and Inspect Raw Data

In [ ]:
RAW_PATH = os.path.join(PROJECT_ROOT, 'data', 'raw', 'energy_weather_raw_data.csv')
df_raw = load_raw_data(RAW_PATH)
print(f'Raw shape: {df_raw.shape}')
print(f'Raw columns: {list(df_raw.columns)}')
print(f'Date range: {df_raw["date"].min()} to {df_raw["date"].max()}')

## 2. Parse and Sort Timestamps

In [ ]:
df = parse_and_sort_timestamps(df_raw.copy())
print(f'Sorted: {df["date"].is_monotonic_increasing}')
print(f'Cleaned shape: {df.shape}')

## 3. Duplicate Summary

In [ ]:
# Remove exact duplicate rows
df, n_exact = remove_exact_duplicate_rows(df)
print(f'Exact duplicate rows removed: {n_exact}')
print(f'Shape after exact-duplicate removal: {df.shape}')

# Check for remaining duplicate timestamps
dup_ts = inspect_duplicate_timestamps(df)
print(f'\nRemaining duplicate-timestamp records: {len(dup_ts)}')
if len(dup_ts) > 0:
    print('\nDuplicate timestamp records:')
    display(dup_ts)

## 4. Duplicate Timestamp Records

In [ ]:
DUP_PATH = os.path.join(PROJECT_ROOT, 'data', 'interim', 'duplicate_timestamp_records.csv')
if os.path.exists(DUP_PATH):
    dup_saved = pd.read_csv(DUP_PATH)
    print(f'Saved duplicate records: {len(dup_saved)}')
    display(dup_saved)
else:
    print('No duplicate timestamp records file found.')

# Resolve remaining duplicate timestamps (keep first)
if len(dup_ts) > 0:
    df = df.drop_duplicates(subset=['date'], keep='first').reset_index(drop=True)
    print(f'\nShape after duplicate-timestamp resolution: {df.shape}')

## 5. Load Hourly Dataset

In [ ]:
HOURLY_PATH = os.path.join(PROJECT_ROOT, 'data', 'interim', 'hourly_aggregated.csv')
hourly = pd.read_csv(HOURLY_PATH, parse_dates=['date'])
print(f'Hourly dataset shape: {hourly.shape}')
print(f'Columns: {list(hourly.columns)}')

## 6. First 5 Hourly Rows

In [ ]:
hourly.head()

## 7. Last 5 Hourly Rows

In [ ]:
hourly.tail()

## 8. Coverage Statistics

In [ ]:
obs = hourly['n_observations']
cov = hourly['coverage_ratio']
total = len(hourly)

print('OBSERVATION COUNT DISTRIBUTION')
print(f'  Total hourly slots         : {total:,}')
print(f'  60 observations (complete) : {int((obs == 60).sum()):,}  ({100.0*(obs==60).sum()/total:.2f}%)')
print(f'  54-59 observations         : {int(((obs >= 54) & (obs <= 59)).sum()):,}')
print(f'  30-53 observations         : {int(((obs >= 30) & (obs <= 53)).sum()):,}')
print(f'  1-29 observations          : {int(((obs >= 1) & (obs <= 29)).sum()):,}')
print(f'  0 observations (missing)   : {int((obs == 0).sum()):,}')
print(f'  >60 observations (anomaly) : {int((obs > 60).sum()):,}')

print(f'\nCOVERAGE RATIO STATISTICS')
print(f'  Min    : {cov.min():.4f}')
print(f'  Max    : {cov.max():.4f}')
print(f'  Mean   : {cov.mean():.4f}')
print(f'  Median : {cov.median():.4f}')
print(f'  Std    : {cov.std():.4f}')

## 9. Coverage Distribution

In [ ]:
print('COVERAGE QUANTILES')
for q in [0.01, 0.05, 0.10, 0.25, 0.50, 0.75, 0.90, 0.95, 0.99]:
    print(f'  {q*100:5.1f}th percentile : {cov.quantile(q):.4f}')

print(f'\nOBSERVATION COUNT VALUE COUNTS (bottom 20):')
vc = obs.value_counts().sort_index()
print(vc.head(20).to_string())

## 10. Candidate Threshold Counts

In [ ]:
print('CANDIDATE THRESHOLD ANALYSIS')
for threshold in [0.80, 0.90, 0.95]:
    qualifying = int((cov >= threshold).sum())
    print(f'  >= {threshold*100:.0f}% coverage : {qualifying:,} hours qualify '
          f'({100.0*qualifying/total:.2f}% of total)')

## 11. Hours with >60 Observations

In [ ]:
over60 = hourly[hourly['n_observations'] > 60]
if len(over60) > 0:
    print(f'WARNING: {len(over60)} hours have >60 observations:')
    display(over60[['date', 'n_observations']])
else:
    print('No hours with >60 observations. PASS.')

## 12. Missing Values in Hourly Dataset

In [ ]:
missing = hourly.isnull().sum()
missing_any = missing[missing > 0]
if len(missing_any) > 0:
    print('Columns with missing values:')
    for col, cnt in missing_any.items():
        print(f'  {col}: {cnt} ({100.0*cnt/total:.2f}%)')
    print(f'\nThese correspond to the {int((obs==0).sum())} hours with 0 observations.')
    print('No interpolation or filling was applied.')
else:
    print('No missing values in the hourly dataset.')

## 13. Step 3.6: Forecasting Setup and Valid-Hour Marking

Threshold: coverage_ratio >= 0.90.


In [ ]:
from src.data.split_data import mark_valid_hours, identify_continuous_runs, get_run_statistics
from src.data.create_sequences import audit_forecasting_windows, create_forecast_windows, validate_window_index

forecast_hourly = mark_valid_hours(hourly, threshold=0.90)
valid_cnt = int(forecast_hourly["is_valid_hour"].sum())
invalid_cnt = len(forecast_hourly) - valid_cnt
print(f"Total hours: {len(forecast_hourly):,}")
print(f"Valid hours (>=90%): {valid_cnt:,} ({100.0*valid_cnt/len(forecast_hourly):.2f}%)")
print(f"Invalid hours (<90%): {invalid_cnt:,} ({100.0*invalid_cnt/len(forecast_hourly):.2f}%)")


## 14. Continuous Run Analysis


In [ ]:
forecast_hourly = identify_continuous_runs(forecast_hourly)
run_stats = get_run_statistics(forecast_hourly)
print("Run Statistics:")
for k, v in run_stats.items():
    if k != "run_length_distribution":
        print(f"  {k}: {v}")


## 15. Forecasting Window Index Construction & Validation

168h history context + 24h forecast horizon = 192h window size.


In [ ]:
window_audit = audit_forecasting_windows(forecast_hourly)
print("Window Audit:")
for k, v in window_audit.items():
    print(f"  {k}: {v}")

window_df = create_forecast_windows(forecast_hourly)
print(f"\nConstructed windows: {len(window_df):,}")
display(window_df.head())

# Validate windows
errors = validate_window_index(window_df, forecast_hourly)
print(f"Validation errors: {len(errors)}")
assert len(errors) == 0, f"Validation failed: {errors}"
print("All window integrity checks PASSED.")


## 16. Future-Weather & Leakage Verification


In [ ]:
forbidden = ["temp_t+1", "feels_like_t+1"]
for col in forbidden:
    assert col not in forecast_hourly.columns, f"LEAKAGE in forecast_hourly: {col}"
    assert col not in window_df.columns, f"LEAKAGE in window_df: {col}"
print("LEAKAGE CHECK PASSED: Zero future-weather columns present.")


## 17. Step 3.7: Chronological Splitting & Validation

Partitioning 5,871 windows chronologically into:
- TRAIN (~65%)
- VALIDATION (~15%)
- CALIBRATION (~10%)
- TEST (~10%)


In [ ]:
from src.data.split_data import (
    load_window_index, sort_windows_chronologically,
    create_chronological_split, validate_split_order,
    audit_split_boundaries
)

# 1. Load and sort window index
win_path = os.path.join(PROJECT_ROOT, "data", "interim", "window_index.csv")
win_df = load_window_index(win_path)
win_df = sort_windows_chronologically(win_df)
total_win = len(win_df)
print(f"Total windows: {total_win:,}")
assert total_win == 5871, f"Expected 5871, got {total_win}"

# 2. Chronological split (65/15/10/10)
splits = create_chronological_split(win_df, 0.65, 0.15, 0.10, 0.10)

print("\nSplit Counts & Percentages:")
for name, s in splits.items():
    cnt = len(s)
    pct = 100.0 * cnt / total_win
    print(f"  {name.upper():<12}: {cnt:>5,} ({pct:>5.2f}%) | "
          f"Origin: {s['forecast_origin'].min()} to {s['forecast_origin'].max()}")

assert len(splits["train"]) == 3816
assert len(splits["validation"]) == 881
assert len(splits["calibration"]) == 587
assert len(splits["test"]) == 587
assert sum(len(s) for s in splits.values()) == total_win
print("\n[PASS] Split counts and total match exactly.")


## 18. Chronological Order & Disjointness Validation

Strict assertion: max(TRAIN) < min(VAL) < min(CAL) < min(TEST).

In [ ]:
# Validate ordering and disjointness
order_errors = validate_split_order(splits)
assert len(order_errors) == 0, f"Split order errors: {order_errors}"

t_max = splits["train"]["forecast_origin"].max()
v_min = splits["validation"]["forecast_origin"].min()
v_max = splits["validation"]["forecast_origin"].max()
c_min = splits["calibration"]["forecast_origin"].min()
c_max = splits["calibration"]["forecast_origin"].max()
test_min = splits["test"]["forecast_origin"].min()

assert t_max < v_min, f"TRAIN does not precede VALIDATION: {t_max} >= {v_min}"
assert v_max < c_min, f"VALIDATION does not precede CALIBRATION: {v_max} >= {c_min}"
assert c_max < test_min, f"CALIBRATION does not precede TEST: {c_max} >= {test_min}"

print(f"TRAIN max origin       ({t_max}) <")
print(f"VALIDATION min origin  ({v_min})")
print(f"VALIDATION max origin  ({v_max}) <")
print(f"CALIBRATION min origin ({c_min})")
print(f"CALIBRATION max origin ({c_max}) <")
print(f"TEST min origin        ({test_min})")
print("\n[PASS] Strict chronological progression: TRAIN < VALIDATION < CALIBRATION < TEST.")

# Disjoint window ID check
all_ids = []
for s in splits.values():
    all_ids.extend(s["window_id"].tolist())
assert len(all_ids) == len(set(all_ids)), "Duplicate window IDs across splits!"
print("[PASS] Zero duplicate window IDs across partitions.")


## 19. Boundary Overlap & Leakage Audit

Verify that the TEST target period is completely unobserved during training and validation.

In [ ]:
audit_res = audit_split_boundaries(splits)
overlaps = audit_res["overlap_audit"]

print("Boundary Overlaps:")
print(f"  Val history overlaps Train target: {overlaps['val_history_overlaps_train_target']}")
print(f"  Cal history overlaps Val target  : {overlaps['cal_history_overlaps_val_target']}")
print(f"  Test history overlaps Cal target : {overlaps['test_history_overlaps_cal_target']}")

# CRITICAL LEAKAGE AUDIT
assert not overlaps["test_target_leaks_into_train"], "CRITICAL: Test target leaks into Train!"
assert not overlaps["test_target_leaks_into_val"], "CRITICAL: Test target leaks into Validation!"
print("\n[PASS] CRITICAL LEAKAGE AUDIT PASSED: ZERO leakage into training or validation.")


## 20. Step 3.8: Leakage-Safe Numerical Feature Scaling

Fitted exclusively on unique valid training observations:
- Energy Scaler: `hourly_active_power`
- Weather Scaler: `temp`, `feels_like`, `humidity`, `pressure`, `speed`, `wind_sin`, `wind_cos`
- Fit on train only: zero leakage into validation, calibration, or test.

In [ ]:
from src.data.scale_data import (
    load_split_boundaries, get_training_hourly_region,
    fit_energy_scaler, fit_weather_scaler, transform_partition,
    load_scalers, PRIMARY_ENERGY_FEATURES, PRIMARY_WEATHER_FEATURES
)

# 1. Scaler Configuration & Fit Boundaries
splits_dir = os.path.join(PROJECT_ROOT, "data", "interim", "splits")
boundaries = load_split_boundaries(splits_dir)
train_start, train_end = boundaries["train"]

hourly_path = os.path.join(PROJECT_ROOT, "data", "interim", "forecast_ready_hourly.csv")
hourly_df = pd.read_csv(hourly_path, parse_dates=["date"])
train_obs = get_training_hourly_region(hourly_df, train_start, train_end, valid_only=True)

print(f"Training Fit Start: {train_start}")
print(f"Training Fit End  : {train_end}")
print(f"Unique Valid Training Observations: {len(train_obs):,}")
assert len(train_obs) == 6943, f"Expected 6943, got {len(train_obs)}"


## 21. Training Scaling & Parameter Inspection

Verify fitted means and standard deviations.

In [ ]:
scalers_dir = os.path.join(PROJECT_ROOT, "outputs", "models", "scalers")
energy_scaler, weather_scaler = load_scalers(scalers_dir)

print("Energy Scaler Mean & Std:")
for feat, m, s in zip(PRIMARY_ENERGY_FEATURES, energy_scaler.mean_, energy_scaler.scale_):
    print(f"  {feat:<20}: mean={m:10.4f}, std={s:10.4f}")

print("\nWeather Scaler Means & Stds:")
for feat, m, s in zip(PRIMARY_WEATHER_FEATURES, weather_scaler.mean_, weather_scaler.scale_):
    print(f"  {feat:<20}: mean={m:10.4f}, std={s:10.4f}")


## 22. Scaled Partition Statistics Audit

Strict assertion: Only TRAIN has mean ≈ 0 and std ≈ 1.

In [ ]:
scaled_dir = os.path.join(PROJECT_ROOT, "data", "processed", "scaled")
all_feats = PRIMARY_ENERGY_FEATURES + PRIMARY_WEATHER_FEATURES

for split_name in ["train", "validation", "calibration", "test"]:
    fpath = os.path.join(scaled_dir, f"{split_name}_scaled.csv")
    sdf = pd.read_csv(fpath, parse_dates=["date"])
    valid_sdf = sdf[sdf["is_valid_hour"]]
    print(f"\n--- {split_name.upper()} ({len(valid_sdf):,} valid rows) ---")
    for feat in all_feats:
        m = valid_sdf[feat].mean()
        s = valid_sdf[feat].std(ddof=0)
        print(f"  {feat:<20}: mean={m:>8.4f}, std={s:>7.4f}")
        if split_name == "train":
            assert np.isclose(m, 0.0, atol=1e-5), f"Train mean not 0 for {feat}"
            assert np.isclose(s, 1.0, atol=1e-5), f"Train std not 1 for {feat}"

print("\n[PASS] Only TRAIN has mean=0.0 and std=1.0. Distribution shifts in val/cal/test preserved.")


## 23. Numerical Inverse-Transform & Leakage Checks

Verify round-trip reconstruction precision (< 1e-10) and absence of forbidden columns.

In [ ]:
# 1. Round-trip check on train sample
train_scaled_df = pd.read_csv(os.path.join(scaled_dir, "train_scaled.csv"))
valid_t_scaled = train_scaled_df[train_scaled_df["is_valid_hour"]]

rec_energy = energy_scaler.inverse_transform(valid_t_scaled[PRIMARY_ENERGY_FEATURES])
rec_weather = weather_scaler.inverse_transform(valid_t_scaled[PRIMARY_WEATHER_FEATURES])

diff_e = np.max(np.abs(rec_energy - train_obs[PRIMARY_ENERGY_FEATURES].values))
diff_w = np.max(np.abs(rec_weather - train_obs[PRIMARY_WEATHER_FEATURES].values))
max_diff = max(diff_e, diff_w)

print(f"Max Reconstruction Error: {max_diff:.2e}")
assert max_diff < 1e-10, f"Reconstruction error too high: {max_diff}"
print("[PASS] Numerical round-trip reconstruction check PASSED (< 1e-10).")

# 2. Forbidden column checks
forbidden = ["temp_t+1", "feels_like_t+1", "current", "voltage", "main", "description"]
for s_name in ["train", "validation", "calibration", "test"]:
    fpath = os.path.join(scaled_dir, f"{s_name}_scaled.csv")
    sdf = pd.read_csv(fpath, nrows=5)
    for col in forbidden:
        assert col not in sdf.columns, f"Forbidden column {col} found in {s_name}_scaled.csv!"
print("[PASS] Zero forbidden / secondary / future columns in scaled datasets.")


## 24. Step 3.9: Model-Ready Sequence Construction & Verification

Verify shape, sample counts, numerical bounds, and absence of NaNs/infs for:
- `X_energy`: `[N, 168, 1]`
- `X_weather`: `[N, 168, 7]`
- `y`: `[N, 24]`


In [ ]:
from src.data.create_sequences import load_partition_sequences

seq_base = os.path.join(PROJECT_ROOT, "data", "processed", "sequences")
expected_shapes = {
    "train": (3816, 168, 1, 7, 24),
    "validation": (881, 168, 1, 7, 24),
    "calibration": (587, 168, 1, 7, 24),
    "test": (587, 168, 1, 7, 24),
}

total_sequences = 0
for s_name, (n, h, e_f, w_f, horiz) in expected_shapes.items():
    p_dir = os.path.join(seq_base, s_name)
    xe, xw, y, meta = load_partition_sequences(p_dir)
    print(f"[{s_name.upper()}]")
    print(f"  X_energy : {xe.shape} | dtype={xe.dtype}")
    print(f"  X_weather: {xw.shape} | dtype={xw.dtype}")
    print(f"  y        : {y.shape}  | dtype={y.dtype}")
    
    assert xe.shape == (n, h, e_f), f"Shape mismatch in {s_name} X_energy"
    assert xw.shape == (n, h, w_f), f"Shape mismatch in {s_name} X_weather"
    assert y.shape == (n, horiz), f"Shape mismatch in {s_name} y"
    assert not np.isnan(xe).any() and not np.isinf(xe).any(), f"NaN/Inf in {s_name} X_energy"
    assert not np.isnan(xw).any() and not np.isinf(xw).any(), f"NaN/Inf in {s_name} X_weather"
    assert not np.isnan(y).any() and not np.isinf(y).any(), f"NaN/Inf in {s_name} y"
    total_sequences += len(xe)

assert total_sequences == 5871
print(f"\n[PASS] All {total_sequences:,} sequences verified with exact shapes and zero NaNs/infs.")


## 25. Inverse-Scale Sanity Verification

De-normalize target predictions to verify plausibility in Watts.

In [ ]:
import joblib
es_path = os.path.join(PROJECT_ROOT, "outputs", "models", "scalers", "energy_scaler.pkl")
energy_scaler = joblib.load(es_path)

print("Target Sanity Check (Inverse-Transformed to Watts):")
for s_name in expected_shapes.keys():
    _, _, y, _ = load_partition_sequences(os.path.join(seq_base, s_name))
    y_orig = energy_scaler.inverse_transform(y.reshape(-1, 1)).reshape(y.shape)
    y_min, y_max, y_mean = y_orig.min(), y_orig.max(), y_orig.mean()
    print(f"  {s_name.upper():<12}: min = {y_min:6.2f} W | max = {y_max:6.2f} W | mean = {y_mean:6.2f} W")
    assert y_min > 0, f"Negative active power in {s_name}"
    assert y_max < 2000, f"Unrealistically high active power in {s_name}"

print("\n[PASS] All targets correspond to plausible household active power consumption in Watts.")
